# GPU-Accelerated Feature Normalization Pipeline
## CERN Higgs Boson Classification

**Goal:** Show that a custom CUDA kernel normalizes 250k x 30 features significantly faster than PyTorch on CPU.

**Pipeline:**
```
Raw CERN Data
   -> Clean (-999 missing values -> per-feature mean)
   -> CPU Normalization Baseline (what we are beating)
   -> Custom CUDA Kernel (2D grid, coalesced memory access)
   -> CUDA Streams (async pipelining of data transfer)
   -> PyTorch MLP Classifier
   -> AUC-ROC evaluation
```

**Hardware:** Google Colab T4 GPU  
**Dataset:** Kaggle Higgs Boson Machine Learning Challenge (2014), 250,000 events

---
## SESSION SETUP (Run This First Every Session)

Run these cells at the start of every Colab session in order:
1. Check GPU
2. Mount Drive + load clean data
3. Build tensors + compute mean/std
4. Compile CUDA kernels
5. Load CUDA libraries

After these 5 cells, all sections below will work without dependency errors.

In [ ]:
# SETUP CELL 1: Verify GPU
# If this says 'no devices found': Runtime > Change Runtime Type > T4 GPU
!nvidia-smi | grep -E 'Tesla|T4|Driver|CUDA'

| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |


In [4]:
# SETUP CELL 2: Mount Drive and load clean dataset
# Requires training.csv already downloaded (see Section 1 for first-time setup)
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/HIGGS_Kaggle'

import os
if os.path.exists(f'{DRIVE_PATH}/X_clean.csv'):
    # Fast path: cleaned data already exists
    X = pd.read_csv(f'{DRIVE_PATH}/X_clean.csv')
    y = pd.read_csv(f'{DRIVE_PATH}/y_clean.csv').squeeze()
    print("Loaded cleaned data from Drive.")
else:
    # First time: load raw and clean
    df = pd.read_csv(f'{DRIVE_PATH}/training.csv')
    df = df.drop(columns=['EventId', 'Weight'])
    df['Label'] = df['Label'].map({'s': 1, 'b': 0})
    feature_cols = [c for c in df.columns if c != 'Label']
    for col in feature_cols:
        valid_mean = df[col][df[col] != -999.0].mean()
        df[col] = df[col].replace(-999.0, valid_mean)
    X = df.drop(columns=['Label'])
    y = df['Label']
    X.to_csv(f'{DRIVE_PATH}/X_clean.csv', index=False)
    y.to_csv(f'{DRIVE_PATH}/y_clean.csv', index=False)
    print("Cleaned and saved to Drive for next time.")

print(f"X: {X.shape} | y: {y.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded cleaned data from Drive.
X: (250000, 30) | y: (250000,)


In [5]:
# SETUP CELL 3: Build tensors and compute mean/std
# Everything downstream depends on X_tensor, mean, std
import torch

X_np = X.to_numpy()
X_tensor = torch.from_numpy(X_np).float()  # zero-copy, CPU, float32

mean = X_tensor.mean(dim=0)  # per-feature mean, shape [30]
std  = X_tensor.std(dim=0)   # per-feature std,  shape [30]

# CPU normalized tensor (used as input to MLP)
X_normalized_cpu = (X_tensor - mean) / std

N, F = X_tensor.shape
print(f"Tensor ready: {X_tensor.shape} | mean: {mean.shape} | std: {std.shape}")

Tensor ready: torch.Size([250000, 30]) | mean: torch.Size([30]) | std: torch.Size([30])


In [2]:
# SETUP CELL 4: Write and compile CUDA kernels
# Colab loses compiled files on disconnect, so we recompile every session
# Takes about 10 seconds

# --- Kernel 1: normalize.cu ---
kernel_code = '''
#include <cuda_runtime.h>
#include <stdio.h>

__global__ void normalize_kernel(
    float* data, float* means, float* stds, int N, int F
) {
    int feature = blockIdx.x * blockDim.x + threadIdx.x;
    int row     = blockIdx.y * blockDim.y + threadIdx.y;
    if (feature >= F || row >= N) return;
    int idx = row * F + feature;
    data[idx] = (data[idx] - means[feature]) / stds[feature];
}

// Full pipeline (alloc + copy + compute + copy back)
extern "C" void run_normalize(
    float* h_data, float* h_means, float* h_stds, int N, int F
) {
    float *d_data, *d_means, *d_stds;
    cudaMalloc(&d_data,  sizeof(float) * N * F);
    cudaMalloc(&d_means, sizeof(float) * F);
    cudaMalloc(&d_stds,  sizeof(float) * F);
    cudaMemcpy(d_data,  h_data,  sizeof(float) * N * F, cudaMemcpyHostToDevice);
    cudaMemcpy(d_means, h_means, sizeof(float) * F,     cudaMemcpyHostToDevice);
    cudaMemcpy(d_stds,  h_stds,  sizeof(float) * F,     cudaMemcpyHostToDevice);

    dim3 block(32, 32);
    dim3 grid((F + 31) / 32, (N + 31) / 32);
    normalize_kernel<<<grid, block>>>(d_data, d_means, d_stds, N, F);
    cudaDeviceSynchronize();

    cudaMemcpy(h_data, d_data, sizeof(float) * N * F, cudaMemcpyDeviceToHost);
    cudaFree(d_data); cudaFree(d_means); cudaFree(d_stds);
}

// Pure compute only: operates directly on GPU memory
extern "C" void run_normalize_device(
    float* d_data, float* d_means, float* d_stds, int N, int F
) {
    dim3 block(32, 32);
    dim3 grid((F + 31) / 32, (N + 31) / 32);
    normalize_kernel<<<grid, block>>>(d_data, d_means, d_stds, N, F);
    cudaDeviceSynchronize();
}
'''

streams_code = '''
#include <cuda_runtime.h>
#include <stdio.h>

__global__ void normalize_kernel(
    float* data, float* means, float* stds, int N, int F
) {
    int feature = blockIdx.x * blockDim.x + threadIdx.x;
    int row     = blockIdx.y * blockDim.y + threadIdx.y;
    if (feature >= F || row >= N) return;
    int idx = row * F + feature;
    data[idx] = (data[idx] - means[feature]) / stds[feature];
}

extern "C" void run_normalize_streams(
    float* h_data, float* h_means, float* h_stds, int N, int F
) {
    int half = N / 2;
    float *d_data0, *d_data1, *d_means, *d_stds;
    cudaMalloc(&d_data0, sizeof(float) * half * F);
    cudaMalloc(&d_data1, sizeof(float) * half * F);
    cudaMalloc(&d_means, sizeof(float) * F);
    cudaMalloc(&d_stds,  sizeof(float) * F);
    cudaMemcpy(d_means, h_means, sizeof(float) * F, cudaMemcpyHostToDevice);
    cudaMemcpy(d_stds,  h_stds,  sizeof(float) * F, cudaMemcpyHostToDevice);
    cudaStream_t stream0, stream1;
    cudaStreamCreate(&stream0);
    cudaStreamCreate(&stream1);
    dim3 block(32, 32);
    dim3 grid((F + 31) / 32, (half + 31) / 32);
    cudaMemcpyAsync(d_data0, h_data, sizeof(float)*half*F, cudaMemcpyHostToDevice, stream0);
    normalize_kernel<<<grid, block, 0, stream0>>>(d_data0, d_means, d_stds, half, F);
    cudaMemcpyAsync(h_data, d_data0, sizeof(float)*half*F, cudaMemcpyDeviceToHost, stream0);
    cudaMemcpyAsync(d_data1, h_data+half*F, sizeof(float)*half*F, cudaMemcpyHostToDevice, stream1);
    normalize_kernel<<<grid, block, 0, stream1>>>(d_data1, d_means, d_stds, half, F);
    cudaMemcpyAsync(h_data+half*F, d_data1, sizeof(float)*half*F, cudaMemcpyDeviceToHost, stream1);
    cudaStreamSynchronize(stream0);
    cudaStreamSynchronize(stream1);
    cudaStreamDestroy(stream0); cudaStreamDestroy(stream1);
    cudaFree(d_data0); cudaFree(d_data1); cudaFree(d_means); cudaFree(d_stds);
}
'''

with open('normalize.cu', 'w') as f:
    f.write(kernel_code)

with open('normalize_streams.cu', 'w') as f:
    f.write(streams_code)

import subprocess
r1 = subprocess.run(['nvcc','-O2','-shared','-Xcompiler','-fPIC','-o','normalize.so','normalize.cu'], capture_output=True, text=True)
r2 = subprocess.run(['nvcc','-O2','-shared','-Xcompiler','-fPIC','-o','normalize_streams.so','normalize_streams.cu'], capture_output=True, text=True)

print('normalize.so:        ', 'OK' if r1.returncode == 0 else r1.stderr)
print('normalize_streams.so:', 'OK' if r2.returncode == 0 else r2.stderr)

normalize.so:         OK
normalize_streams.so: OK


In [6]:
# SETUP CELL 5: Load CUDA libraries and prepare ctypes pointers
import ctypes

lib = ctypes.CDLL('./normalize.so')
lib.run_normalize.argtypes = [
    ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float),
    ctypes.c_int, ctypes.c_int,
]
lib.run_normalize.restype = None

lib.run_normalize_device.argtypes = [
    ctypes.c_void_p,
    ctypes.c_void_p,
    ctypes.c_void_p,
    ctypes.c_int,
    ctypes.c_int,
]
lib.run_normalize_device.restype = None

lib_streams = ctypes.CDLL('./normalize_streams.so')
lib_streams.run_normalize_streams.argtypes = [
    ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float),
    ctypes.c_int, ctypes.c_int,
]
lib_streams.run_normalize_streams.restype = None

# Shared ctypes pointers for means and stds (reused across benchmarks)
means_np  = mean.numpy()
stds_np   = std.numpy()
means_ptr = means_np.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
stds_ptr  = stds_np.ctypes.data_as(ctypes.POINTER(ctypes.c_float))

print("All libraries loaded. Session ready.")

All libraries loaded. Session ready.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## Section 1: One-Time Dataset Setup
Run this section only once to download and save the dataset to Google Drive.
After this, Session Setup Cell 2 handles loading automatically.

**Requires:** Kaggle account + kaggle.json API key
Get it at: kaggle.com > Settings > API > Create New Token

**Also requires:** Accepting competition rules at kaggle.com/competitions/higgs-boson/rules

In [ ]:
# ONE-TIME ONLY: Download dataset from Kaggle to Drive
from google.colab import drive, files
drive.mount('/content/drive')

!pip install kaggle -q
files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c higgs-boson -p "/content/drive/MyDrive/HIGGS_Kaggle"

import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/HIGGS_Kaggle/higgs-boson.zip', 'r') as z:
    z.extractall('/content/drive/MyDrive/HIGGS_Kaggle/')
with zipfile.ZipFile('/content/drive/MyDrive/HIGGS_Kaggle/training.zip', 'r') as z:
    z.extractall('/content/drive/MyDrive/HIGGS_Kaggle/')

print("Dataset saved to Drive. You only need to do this once.")

---
## Section 2: Data Inspection
Run if you want to explore the raw dataset. Not required for benchmarking.

In [8]:
# Inspect the loaded data
# X and y are already loaded by Session Setup Cell 2
print("Feature matrix X:", X.shape)
print("Label vector y:  ", y.shape)
print("\nClass distribution:")
print(y.value_counts())
print("\nSample features:")
display(X.head(3))

Feature matrix X: (250000, 30)
Label vector y:   (250000,)

Class distribution:
Label
0    164333
1     85667
Name: count, dtype: int64

Sample features:


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
0,138.470000,51.655,97.827,27.980,0.910000,124.71100,2.666000,3.064,41.928,197.760,...,-0.277,258.733,2,67.435,2.150,0.444,46.062000,1.240000,-2.475000,113.497
1,160.937000,68.768,103.235,48.146,2.403735,371.78336,-0.821688,3.473,2.078,125.157,...,-1.916,164.546,1,46.226,0.725,1.158,57.679474,-0.011845,-0.001582,46.226
2,121.858528,162.172,125.953,35.635,2.403735,371.78336,-0.821688,3.148,9.336,197.814,...,-2.186,260.414,1,44.251,2.053,-2.028,57.679474,-0.011845,-0.001582,44.251


---
## Section 3: CPU Normalization Baseline

Z-score normalization on CPU using PyTorch. This is the number we are beating.

Formula: `x_normalized = (x - mean) / std`

Timing methodology: warmup run discarded, median of 10 runs reported.

In [15]:
import time, statistics

# Warmup: first call has one-time overhead (memory allocation, cache warmup)
_ = (X_tensor - mean) / std

times = []
for _ in range(10):
    start = time.perf_counter()
    X_normalized_cpu = (X_tensor - mean) / std
    end = time.perf_counter()
    times.append((end - start) * 1000)

CPU_BASELINE_MS = statistics.median(times)
print(f"All times (ms): {[f'{t:.2f}' for t in times]}")
print(f"Median (official baseline): {CPU_BASELINE_MS:.2f} ms")

All times (ms): ['9.98', '12.64', '18.44', '18.16', '12.13', '18.23', '18.47', '11.89', '17.49', '18.03']
Median (official baseline): 17.76 ms


---
## Section 4: Custom CUDA Kernel Benchmark

Two benchmarks:
1. GPU with full data transfer (CPU -> GPU -> compute -> GPU -> CPU)
2. GPU compute only (data stays on GPU, pure kernel speed)

The gap between these two numbers reveals that data transfer, not computation, is the bottleneck.

In [16]:
# Benchmark 1: GPU with full data transfer
# lib and means_ptr/stds_ptr loaded in Session Setup Cell 5

# Warmup
data_np = X_tensor.numpy().copy()
data_ptr = data_np.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
lib.run_normalize(data_ptr, means_ptr, stds_ptr, N, F)

times = []
for _ in range(10):
    data_np  = X_tensor.numpy().copy()  # reset: kernel modifies in-place
    data_ptr = data_np.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    start = time.perf_counter()
    lib.run_normalize(data_ptr, means_ptr, stds_ptr, N, F)
    end = time.perf_counter()
    times.append((end - start) * 1000)

GPU_KERNEL_MS = statistics.median(times)
print(f"All times (ms): {[f'{t:.2f}' for t in times]}")
print(f"Median (GPU with transfers): {GPU_KERNEL_MS:.2f} ms")
print(f"Speedup vs CPU: {CPU_BASELINE_MS / GPU_KERNEL_MS:.2f}x")

All times (ms): ['14.25', '15.32', '14.82', '15.41', '14.46', '15.27', '14.09', '14.81', '14.59', '15.40']
Median (GPU with transfers): 14.82 ms
Speedup vs CPU: 1.20x


In [17]:
# Benchmark 2: Custom CUDA Kernel (compute only)
device   = torch.device('cuda')
X_gpu    = X_tensor.to(device)
mean_gpu = mean.to(device)
std_gpu  = std.to(device)

d_data_ptr = ctypes.c_void_p(X_gpu.data_ptr())
d_means_ptr = ctypes.c_void_p(mean_gpu.data_ptr())
d_stds_ptr = ctypes.c_void_p(std_gpu.data_ptr())

# Warmup run
lib.run_normalize_device(d_data_ptr, d_means_ptr, d_stds_ptr, N, F)
torch.cuda.synchronize()

times = []
for _ in range(10):
    # Reset device data so values do not explode across iterations
    X_gpu.copy_(X_tensor)
    torch.cuda.synchronize()

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    lib.run_normalize_device(d_data_ptr, d_means_ptr, d_stds_ptr, N, F)
    end_event.record()

    torch.cuda.synchronize()
    times.append(start_event.elapsed_time(end_event))

GPU_COMPUTE_MS = statistics.median(times)
print(f"All times (ms): {[f'{t:.2f}' for t in times]}")
print(f"Median (Custom Kernel compute only): {GPU_COMPUTE_MS:.2f} ms")
print(f"Speedup vs CPU: {CPU_BASELINE_MS / GPU_COMPUTE_MS:.2f}x")

All times (ms): ['0.46', '0.45', '0.44', '0.44', '0.47', '0.45', '0.43', '0.43', '0.43', '0.43']
Median (Custom Kernel compute only): 0.44 ms
Speedup vs CPU: 40.43x


---
## Section 5: CUDA Streams Benchmark

Streams pipeline: overlap data transfer of batch 2 with computation on batch 1.
Requires pinned (page-locked) memory for true async overlap.
Regular numpy arrays use pageable memory and async calls silently become synchronous.

In [18]:
# Pinned memory: required for cudaMemcpyAsync to truly overlap with kernel execution
X_pinned       = X_tensor.pin_memory()
data_np_pinned = X_pinned.numpy()

# Warmup
data_np  = data_np_pinned.copy()
data_ptr = data_np.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
lib_streams.run_normalize_streams(data_ptr, means_ptr, stds_ptr, N, F)

times = []
for _ in range(10):
    data_np  = data_np_pinned.copy()
    data_ptr = data_np.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    start = time.perf_counter()
    lib_streams.run_normalize_streams(data_ptr, means_ptr, stds_ptr, N, F)
    end = time.perf_counter()
    times.append((end - start) * 1000)

GPU_STREAMS_MS = statistics.median(times)
print(f"All times (ms): {[f'{t:.2f}' for t in times]}")
print(f"Median (streams + pinned): {GPU_STREAMS_MS:.2f} ms")
print(f"Speedup vs CPU:            {CPU_BASELINE_MS / GPU_STREAMS_MS:.2f}x")
print(f"Speedup vs no-streams:     {GPU_KERNEL_MS / GPU_STREAMS_MS:.2f}x")

All times (ms): ['14.86', '14.22', '14.84', '14.63', '14.81', '14.09', '15.17', '14.44', '14.84', '13.89']
Median (streams + pinned): 14.72 ms
Speedup vs CPU:            1.21x
Speedup vs no-streams:     1.01x


---
## Section 6: PyTorch MLP Classifier

Architecture: 30 -> 128 -> 64 -> 1 (sigmoid)
Loss: BCELoss | Optimizer: Adam | Metric: AUC-ROC

In [19]:
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

class HiggsClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(30, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

# Train/test split: X_normalized_cpu is already computed in Session Setup Cell 3
X_norm_np = X_normalized_cpu.numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X_norm_np, y.values, test_size=0.2, random_state=42
)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to('cuda')
X_test_t  = torch.tensor(X_test,  dtype=torch.float32).to('cuda')
y_train_t = torch.tensor(y_train, dtype=torch.float32).to('cuda')
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).to('cuda')

model     = HiggsClassifier().to('cuda')
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loader    = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=1024, shuffle=True)

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch).squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}  |  Loss: {total_loss/len(loader):.4f}")

model.eval()
with torch.no_grad():
    y_probs = model(X_test_t).squeeze().cpu().numpy()

auc = roc_auc_score(y_test, y_probs)
print(f"\nValidation AUC-ROC: {auc:.4f}")

Epoch 1/10  |  Loss: 0.4548
Epoch 2/10  |  Loss: 0.3838
Epoch 3/10  |  Loss: 0.3746
Epoch 4/10  |  Loss: 0.3688
Epoch 5/10  |  Loss: 0.3645
Epoch 6/10  |  Loss: 0.3621
Epoch 7/10  |  Loss: 0.3602
Epoch 8/10  |  Loss: 0.3589
Epoch 9/10  |  Loss: 0.3573
Epoch 10/10  |  Loss: 0.3559

Validation AUC-ROC: 0.9087


---
## Section 7: Final Results

In [ ]:
# All benchmark variables are set by running the sections above in order
# If any variable is missing, re-run the corresponding section

print("=" * 55)
print("   CERN HIGGS BOSON CLASSIFICATION - RESULTS")
print("=" * 55)
print(f"\nNORMALIZATION BENCHMARK (250k x 30 features)")
print(f"  CPU baseline:          {CPU_BASELINE_MS:.2f} ms")
print(f"  GPU compute only:      {GPU_COMPUTE_MS:.2f} ms  ({CPU_BASELINE_MS/GPU_COMPUTE_MS:.1f}x speedup)")
print(f"  GPU with transfers:    {GPU_KERNEL_MS:.2f} ms  ({CPU_BASELINE_MS/GPU_KERNEL_MS:.1f}x speedup)")
print(f"  GPU streams + pinned:  {GPU_STREAMS_MS:.2f} ms  ({CPU_BASELINE_MS/GPU_STREAMS_MS:.1f}x speedup)")
print(f"\nMLP CLASSIFIER")
print(f"  Architecture:          30 -> 128 -> 64 -> 1")
print(f"  Epochs:                10")
print(f"  Validation AUC-ROC:    {auc:.4f}")
print(f"  Kaggle winner AUC:     0.8800  (not directly comparable, different test set)")
print("=" * 55)

   CERN HIGGS BOSON CLASSIFICATION - RESULTS

NORMALIZATION BENCHMARK (250k x 30 features)
  CPU baseline:          25.10 ms
  GPU compute only:      0.57 ms  (43.7x speedup)
  GPU with transfers:    13.56 ms  (1.9x speedup)
  GPU streams + pinned:  13.39 ms  (1.9x speedup)

MLP CLASSIFIER
  Architecture:          30 -> 128 -> 64 -> 1
  Epochs:                10
  Validation AUC-ROC:    0.9093
  Kaggle winner AUC:     0.8800  (not directly comparable, different test set)
